# Layerwise Jacobian singular-value distribution for heavy-tailed MLPs

Visualisation of `ht_mlp_jacobian.py` / `ht_mlp_jacobian.md`. Four pathways target the SV density of $J^l = D^l W^l$ at the heavy-tailed MFT fixed point $q^*$:

* **(P1)** Theorem-2 deterministic-quantile theory -- single scalar closure in $Y_r$ (`RMT/one_sided_wishart_levy.py`; dedicated Theorem 2 solver with anchor-first seed continuation, imag-eps homotopy, and *independent*-rule $h_\alpha$ density readout for accuracy at small $s$).
* **(P2)** Population-dynamics cavity-equation route (`RMT.py:jac_cavity_svd_log_pdf`).
* **(P3a)** Synthetic empirical SVDs: $h_j \sim S^* p_\alpha$ directly, $J = D W$, batch SVDs.
* **(P3b)** MLP-derived empirical SVDs: `RMT.MLP` at depth where $q^l \to q^*$.

Each cell times its body; results saved to `fig/ht_mlp_jacobian/` for the panel sweep.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

cwd = Path.cwd()
if (cwd / 'ht_mlp_jacobian.py').exists():
    sys.path.insert(0, str(cwd))
elif (cwd / 'random_dnn' / 'ht_mlp_jacobian.py').exists():
    sys.path.insert(0, str(cwd / 'random_dnn'))

import ht_mlp_jacobian as htj
sys.path.insert(0, str(cwd / 'RMT') if (cwd / 'RMT').is_dir() else str(cwd / 'random_dnn' / 'RMT'))
import one_sided_wishart_levy as osw
import localisation as loc

FIGDIR = cwd / 'fig' / 'ht_mlp_jacobian'
FIGDIR.mkdir(parents=True, exist_ok=True)

## 0. Convention check

Sanity gate: the structured-curve native sampler `swl.sample_structured_levy_matrix(entry_scale=sigma_w)` should produce the same SV histogram as the MLP-style sampler `sigma_w * (2N)**(-1/alpha) * scipy_stable(scale=1)` (Belinschi-vs-SciPy factor absorbed into the prefactor). Pass criterion: agreement within Monte-Carlo bin noise.

In [ ]:
conv = htj.convention_check(alpha=1.5, sigma_w=1.0, N=256, num_matrices=80, bins=121, seed=0)
print(f"max abs density diff: {conv['max_abs_diff']:.4f}")
print(f"relative L1 diff:     {conv['rel_L1_diff']:.4f}")

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(conv['centres'], conv['density_structured_native'], label='structured native')
ax.plot(conv['centres'], conv['density_mlp_style'], '--', label='MLP-style')
ax.set(xlabel='singular value $s$', ylabel='density', title=r'convention check ($\alpha=1.5$, $\sigma_w=1$)')
ax.legend(); fig.tight_layout()
fig.savefig(FIGDIR / 'convention_check.png', dpi=120)
plt.show()

## 1. Three-way validation at a representative point

$(\alpha, \sigma_w) = (1.5, 1.0)$, $\phi = \tanh$. The forward MFT gives $q^* \approx 0.106$, $S^* \approx 0.224$.

All four pathways (P1, P2, P3a, P3b) plotted on the same SV axis.

In [ ]:
out = htj.run_validation(
    alpha=1.5, sigma_W=1.0, N=256, num_matrices=40,
    depth=50, burn_in=25, num_doublings=7, num_chis=1,
    s_max=6.0, num_points=121, bins=61,
    n_profile_samples=30_000, do_mlp=True, seed=0,
    save_path=str(FIGDIR / 'validation_alpha1.5_sigma1.0.npz'),
)

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Linear-scale density comparison
ax1.plot(out['sv_theory'][1:], out['density_theory'][1:],
         color='C0', lw=2, label='(P1) Theorem 2 theory')
ax1.plot(out['sv_grid_pop_dyn'], out['density_pop_dyn'],
         color='C1', lw=1, label='(P2) population dynamics')
ax1.plot(out['sv_synthetic_centres'], out['density_synthetic'],
         'o', color='C2', ms=4, label='(P3a) synthetic empirical')
if len(out['sv_mlp_centres']):
    ax1.plot(out['sv_mlp_centres'], out['density_mlp'],
             's', color='C3', ms=4, mfc='none', label='(P3b) MLP empirical')
ax1.set(xlabel='singular value $s$', ylabel='density',
        title=fr"$\alpha=1.5$, $\sigma_w=1.0$, $q^*={out['summary']['q_star']:.3f}$")
ax1.legend(); ax1.set_xlim(0, 4)

# Tail: log-log with all pathways + B s^{-(1+alpha)} reference
B = out['summary']['tail_constant_B']
s_tail = np.linspace(2.0, 8.0, 50)
ax2.loglog(out['sv_theory'][1:], out['density_theory'][1:],
           color='C0', lw=2, label='(P1) theory')
ax2.loglog(out['sv_grid_pop_dyn'], out['density_pop_dyn'],
           color='C1', lw=1, label='(P2) population dynamics')
ax2.loglog(out['sv_synthetic_centres'], out['density_synthetic'],
           'o', color='C2', ms=4, label='(P3a) synthetic')
if len(out['sv_mlp_centres']):
    ax2.loglog(out['sv_mlp_centres'], out['density_mlp'],
               's', color='C3', ms=4, mfc='none', label='(P3b) MLP empirical')
ax2.loglog(s_tail, B * s_tail ** (-2.5),
           ':', color='k', label=fr'$B\,s^{{-(1+\alpha)}}$, $B={B:.3f}$')
ax2.set(xlabel='singular value $s$', ylabel='density',
        title='heavy tail $f(s) \\sim B\\,s^{-(1+\\alpha)}$',
        ylim=(1e-3, 3.0))
ax2.legend()

fig.tight_layout()
fig.savefig(FIGDIR / 'validation_alpha1.5_sigma1.0.png', dpi=120)
plt.show()
print('summary diffs:')
for k in ('diff_theory_synthetic', 'diff_pop_dyn_synthetic', 'diff_theory_pop_dyn',
         'convention_max_abs_diff', 'q_rel_err_mlp'):
    print(f'  {k}: {out["summary"][k]:.4f}')

## 2. Panel across $(\alpha, \sigma_w)$

The shape evolves as $\alpha$ moves from heavy-tailed ($\alpha=1.3$, fat tail / sharp peak at small $s$) toward Gaussian ($\alpha=1.9$, Marchenko-Pastur-like). $\sigma_w$ controls $q^*$ and hence the column profile $c$.

Each panel: (P1) theory + (P3a) synthetic empirical. The pop-dynamics and MLP-derived routes are omitted for compactness but verified at the representative point above.

In [ ]:
alphas = [1.3, 1.5, 1.7]
sigmas = [0.8, 1.2, 1.8]
fig, axes = plt.subplots(len(alphas), len(sigmas), figsize=(4 * len(sigmas), 2.6 * len(alphas)), sharex=True)
for i, alpha in enumerate(alphas):
    for j, sw in enumerate(sigmas):
        ax = axes[i, j]
        profile = htj.jacobian_profile(alpha, sw, n_samples=15_000, seed=0)
        curve, _ = htj.theoretical_jacobian_sv_curve(
            alpha, sw, s_max=8.0, num_points=61, profile=profile,
            quadrature_order=64, profile_order=48,
        )
        emp = htj.synthetic_jacobian_sv_spectrum(
            alpha, sw, q_star=profile.q_star,
            N=150, num_matrices=15, seed=0, bins=41,
            sv_range=(0.0, 8.0),
        )
        ax.plot(curve.singular_values[1:], curve.singular_density[1:], 'C0', lw=1.6, label='theory')
        ax.plot(emp['centres'], emp['density'], 'o', color='C2', ms=3, label='empirical')
        ax.set_xlim(0, 5)
        ax.set_title(fr"$\alpha={alpha}$, $\sigma_w={sw}$, $q^*={profile.q_star:.2f}$", fontsize=10)
        if i == len(alphas) - 1: ax.set_xlabel(r'$s$')
        if j == 0: ax.set_ylabel('density')
        if i == 0 and j == 0: ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGDIR / 'panel_alpha_sigma.png', dpi=120)
plt.show()

## 3. Column profile $c(v)$ visualisation

The quantile function $c(v) = F^{-1}_{|\phi'(S^* Z)|}(v)$, $Z \sim p_\alpha$, is the deterministic profile fed to Theorem 2.

For $\phi = \tanh$, $|\phi'(h)| = 1 - \tanh^2(h) \in (0, 1]$, with $|\phi'(0)| = 1$ (linear regime) and $|\phi'(|h| \to \infty)| \to 0$ (saturation). Heavy-tailed $h$ at scale $S^*$ has many saturated neurons -> $c(v)$ is small over a wide low-$v$ tail and rises sharply near $v = 1$.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4))
v_grid = np.linspace(0, 1, 401)
for alpha, sw, color in [(1.3, 1.0, 'C0'), (1.5, 1.0, 'C1'), (1.7, 1.0, 'C2'), (1.9, 1.0, 'C3')]:
    profile = htj.jacobian_profile(alpha, sw, n_samples=50_000, seed=0)
    c = htj.quantile_callable(profile)
    ax.plot(v_grid, c(v_grid), color=color,
            label=fr"$\alpha={alpha}$ ($S^*={profile.S_star:.3f}$)")
ax.set(xlabel=r'$v \in [0,1]$', ylabel=r"$c(v) = F^{-1}_{|\phi'(S^* Z)|}(v)$",
       title=r'column profile $c(v)$ at $\sigma_w=1$')
ax.legend(); fig.tight_layout()
fig.savefig(FIGDIR / 'column_profile.png', dpi=120)
plt.show()

## 4. Profile-aligned localisation $\ell_q^{(2)}(s)$ of right singular vectors

The right singular vectors of $J^l = D^l W^l$ inherit the column-profile $c(v) = F^{-1}_{|\phi'(S^* Z)|}(v)$ through the slaved cavity field
$Y_c(y, z) = C_\alpha c(y)^\alpha g_\alpha(Y_r(z)) / [(1+\gamma) z^\alpha]$
(Theorem 2(i) of `RMT/one_sided_wishart_levy.md` / sec. 8 of `RMT/localisation.md`).
Profile-aligned localisation -- right singular vectors concentrating on neurons in particular regimes of $|\phi'|$ -- is captured by the Jensen-gap index

$$
\ell_q^{(2)}(s) \;=\; 1 - \frac{\int_0^1 (\rho_y^{(2)}(s))^q\, dy}{(\int_0^1 \rho_y^{(2)}(s)\, dy)^q}, \qquad \rho_y^{(2)}(s) = -\frac{1}{\pi s}\,\mathrm{Im}\, h_\alpha(Y_c(y, s + i 0^+)),
$$

with $q = \alpha/2$ the natural choice. $\ell_q^{(2)} \in [0, 1]$ measures the Jensen-concavity gap of the per-column LDoS distribution across $y$; $\ell_q^{(2)} = 0$ iff $\rho_y^{(2)}$ is uniform in $y$ (profile-delocalised right vectors). The left-side $\ell_q^{(1)} \equiv 0$ identically in the one-sided case since $Y_r$ is $x$-independent.

This is the BDG-deterministic-limit, profile-alignment piece -- *distinct* from the multifractal exponent $D_q$ (which lives in the $\log N$-slope and requires finite-$N$ heavy-tail fluctuations outside the BDG limit).

In [ ]:
# Theory localisation index curve
loc = loc.localisation_index_curve(
    alpha=1.5, gamma=1.0, c=htj.quantile_callable(htj.jacobian_profile(1.5, 1.0, n_samples=30_000, seed=0)),
    q=0.75,
    s_max=4.0, num_points=81, imag_eps=1e-3,
    entry_scale=1.0, normalization='stable',
    quadrature_order=96, profile_order=64,
)

# Empirical localisation index from SVD of column-scaled heavy-tailed matrices
emp_loc = loc.empirical_localisation_index_from_svd(
    alpha=1.5, gamma=1.0, c=htj.quantile_callable(htj.jacobian_profile(1.5, 1.0, n_samples=30_000, seed=0)),
    q=0.75,
    n_rows=200, num_matrices=60, sv_bins=20, sv_range=(0.0, 4.0), seed=0,
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

# Left: ell_q(s) theory vs empirical
ax1.plot(loc.singular_values[1:], loc.ell_q_col[1:], color='C0', lw=2, label=fr'theory ($q={loc.q:.2g}$)')
ax1.plot(emp_loc['sv_centres'], emp_loc['ell_q_col'], 'o', color='C2', ms=5, label='empirical (SVD)')
ax1.axhline(0, color='k', lw=0.5, alpha=0.5)
ax1.set(xlabel=r'$s$', ylabel=r'$\ell_q^{(2)}(s)$',
        title=r'profile-aligned right-vec localisation, $\alpha=1.5$, $\sigma_w=1.0$')
ax1.legend()

# Right: per-column LDoS heatmap rho^(2)_y(s) (theory)
im = ax2.imshow(
    loc.per_column_ldos[1:, :].T,
    aspect='auto',
    origin='lower',
    extent=[loc.singular_values[1], loc.singular_values[-1], 0.0, 1.0],
    cmap='viridis',
)
ax2.set(xlabel=r'$s$', ylabel=r'column position $y$',
        title=r'per-column LDoS $\rho_y^{(2)}(s)$ (theory)')
fig.colorbar(im, ax=ax2, label=r'$\rho_y^{(2)}$')

fig.tight_layout()
fig.savefig(FIGDIR / 'localisation_alpha1.5_sigma1.0.png', dpi=120)
plt.show()
print(f'ell_q max (theory):     {np.nanmax(loc.ell_q_col):.4f} at s={loc.singular_values[np.nanargmax(loc.ell_q_col)]:.3f}')
print(f'ell_q max (empirical): {np.nanmax(emp_loc["ell_q_col"]):.4f} at s={emp_loc["sv_centres"][np.nanargmax(emp_loc["ell_q_col"])]:.3f}')

## 5. Empirical $D_q$ (multifractal exponent) of left/right singular vectors

For each MLP realisation, the right and left singular vectors of $J^l = D^l W^l$ at post-burn-in layers carry a per-vector multifractal exponent $D_q = -\log I_q / ((q-1) \log N)$ with $I_q = \sum_i |v(i)|^{2q}$.  This is the classical IPR-based fractal dimension; well-defined and observable for both left ($u_k$) and right ($v_k$) singular vectors at finite $N$ regardless of any rigorous theoretical pairing (which is open -- see `RMT/localisation.md` sec. 9).

Below: $D_q^{\text{left}}(s)$ and $D_q^{\text{right}}(s)$ binned by singular value $s$, at $(\alpha, \sigma_w) = (1.5, 1.0)$, for a few $q$ values.  Pure empirics, no theory curve overlaid.

In [ ]:
q_grid = np.array([0.5, 1.0, 1.5, 2.0])
dq_emp = htj.mlp_jacobian_Dq_spectrum(
    alpha=1.5, sigma_w=1.0, q_grid=q_grid,
    N=256, depth=50, num_matrices=12, burn_in=25,
    sv_bins=18, sv_range=(0.0, 4.0), seed=0,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for qi, q in enumerate(q_grid):
    color = plt.cm.viridis(qi / max(1, len(q_grid) - 1))
    axes[0].plot(dq_emp['sv_centres'], dq_emp['Dq_left_mean'][:, qi],
                 'o-', color=color, ms=4, label=fr'$q={q}$')
    axes[1].plot(dq_emp['sv_centres'], dq_emp['Dq_right_mean'][:, qi],
                 's-', color=color, ms=4, label=fr'$q={q}$')
for ax, title in zip(axes, ['left singular vectors $u_k$', 'right singular vectors $v_k$']):
    ax.set(xlabel=r'$s$', title=title, ylim=(0, 1.05))
    ax.axhline(1.0, color='k', lw=0.5, ls='--', alpha=0.5)
    ax.legend(fontsize=9, ncol=2)
axes[0].set_ylabel(r'$D_q(s)$ (empirical, finite-$N$)')
fig.suptitle(fr'Empirical multifractal $D_q$ of postjac SVs at $\alpha=1.5$, $\sigma_w=1.0$, $N={dq_emp["N"]}$', y=1.02)
fig.tight_layout()
fig.savefig(FIGDIR / 'Dq_alpha1.5_sigma1.0.png', dpi=120)
plt.show()

print(f'bin counts: {dq_emp["bin_count"]}')
print(f'$D_2$ left  range: {np.nanmin(dq_emp["Dq_left_mean"][:, -1]):.3f} - {np.nanmax(dq_emp["Dq_left_mean"][:, -1]):.3f}')
print(f'$D_2$ right range: {np.nanmin(dq_emp["Dq_right_mean"][:, -1]):.3f} - {np.nanmax(dq_emp["Dq_right_mean"][:, -1]):.3f}')

## 6. Verifying the analytical $D_q$ formula against direct SVD-IPR

Section 9H of `RMT/localisation.md` derives
$$
D_q(\lambda) = 1 - \frac{1}{(q-1)\log N}\,\log\!\left[\frac{M_q(\lambda)}{(\pi\rho(\lambda))^q}\right], \qquad M_q := \mathbb{E}_{\mu^z}\!\left[(-\mathrm{Im}\,G)^q\right]
$$
from the BG spectral identity $|v(i)|^2 = -\mathrm{Im}\,G_{ii}/(N\pi\rho)$.  To verify the formula at the level of finite-$N$ matrices, we compute *both estimators on the same MLP realisations*:

- **Direct IPR**: $D_q^{\text{IPR}}(s) = -\log \bar I_q / ((q-1)\log N)$ from $I_q = \sum_i |v(i)|^{2q}$ per eigenvector, averaged over the SV bin (gold standard).
- **Formula**: $D_q^{\text{form}}(s)$ via the equation above, with $M_q = (1/N)\sum_i (-\mathrm{Im}\,G_{ii}(s + i\eta))^q$ computed from the SVD output by spectral decomposition.

Both come from the same matrices.  Agreement verifies the formula's underlying self-averaging identity at finite $N$.

In [ ]:
dq_check = htj.verify_Dq_formula(
    alpha=1.5, sigma_w=1.0, q_grid=q_grid,
    N=256, depth=50, num_matrices=10, burn_in=25,
    sv_bins=18, sv_range=(0.0, 4.0), seed=0,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4), sharey=True)
for qi, q in enumerate(q_grid):
    color = plt.cm.viridis(qi / max(1, len(q_grid) - 1))
    axes[0].plot(dq_check['sv_centres'], dq_check['Dq_ipr_left'][:, qi],
                 'o', color=color, ms=5, label=fr'IPR  $q={q}$')
    axes[0].plot(dq_check['sv_centres'], dq_check['Dq_form_left'][:, qi],
                 '-', color=color, lw=1.5, label=fr'form $q={q}$')
    axes[1].plot(dq_check['sv_centres'], dq_check['Dq_ipr_right'][:, qi],
                 's', color=color, ms=5)
    axes[1].plot(dq_check['sv_centres'], dq_check['Dq_form_right'][:, qi],
                 '-', color=color, lw=1.5)
for ax, title in zip(axes, ['left SVs $u_k$', 'right SVs $v_k$']):
    ax.set(xlabel=r'$s$', title=title)
    ax.axhline(1.0, color='k', lw=0.5, ls='--', alpha=0.5)
    ax.legend(fontsize=8, ncol=2)
axes[0].set_ylabel(r'$D_q(s)$')
fig.suptitle(fr'D_q: direct IPR (markers) vs formula (lines, eq. 14), same matrices; $\alpha=1.5$, $\sigma_w=1$, $N={dq_check["N"]}$, $\eta=1/N$', y=1.03)
fig.tight_layout()
fig.savefig(FIGDIR / 'Dq_formula_vs_IPR_alpha1.5_sigma1.0.png', dpi=120)
plt.show()

print(f'eta = {dq_check["eta"]:.4f}')
# Mean gap formula - IPR per q
for qi, q in enumerate(q_grid):
    if abs(q - 1) < 1e-12: continue
    gap_l = np.nanmean(dq_check['Dq_form_left'][:, qi] - dq_check['Dq_ipr_left'][:, qi])
    gap_r = np.nanmean(dq_check['Dq_form_right'][:, qi] - dq_check['Dq_ipr_right'][:, qi])
    print(f'  q={q}: mean (form - IPR)  left={gap_l:+.3f}  right={gap_r:+.3f}')

## 7. Localisation onset via density-deviation diagnostic

Following sec. 9J of `RMT/localisation.md`: compare the analytical SV density (BDG field theory, Belinschi $h_\alpha$ readout) to the population-dynamics estimate (cavity RDE pool-mean of $\mathrm{Im}\,g$) at fixed pool size, on a fine SV grid.  Systematic deviation beyond Monte-Carlo noise signals heavy-tail fluctuations of the local resolvent across the pool -- the Aizenman-Molchanov / BG fractional-moment localisation signature recast at the density level.

**Hermitisation mapping (correspondence to BG regimes):** for rectangular SVs at stability index $\alpha_\text{SV}$, the bipartite Hermitisation cavity self-energy involves squared entries with stability index $\alpha_\text{SV}/2$.  So:
- $\alpha_\text{SV} = 1.5 \;\Rightarrow\; \alpha_\text{Wigner-equiv} = 0.75$, in BG's *unresolved intermediate regime* $2/3 \le \alpha \le 1$.
- $\alpha_\text{SV} < 4/3 \;\Rightarrow\; \alpha_\text{Wigner-equiv} < 2/3$, BG's *proven localised* regime.

Our operational regime $\alpha_\text{SV} \in (1, 2)$ straddles the BG transition through Hermitisation.  The diagnostic should pick this up as deviation onset in the SV spectral tail.

In [ ]:
dev = htj.density_deviation_diagnostic(
    alpha=1.5, sigma_w=1.0,
    s_max=5.0, num_points=81,
    num_doublings=8, num_chis=8,
    quadrature_order=96, profile_order=64,
    seed=0,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

# Left: rho_thy and rho_popdyn overlaid
ax = axes[0]
ax.plot(dev['sv_grid'][1:], dev['rho_thy'][1:], '-', color='C0', lw=2, label='analytical (Belinschi)')
ax.errorbar(dev['sv_grid'][1:], dev['rho_popdyn'][1:], yerr=dev['rho_popdyn_std'][1:],
            fmt='o', color='C1', ms=3, ecolor='C1', alpha=0.6, label='popdyn (cavity RDE)')
ax.set(xlabel=r'$s$', ylabel=r'$\rho(s)$',
       title=fr'analytical vs popdyn density, $\alpha=1.5$, $\sigma_w=1$, pool$=2^{{{dev["num_doublings"]}}}={dev["pool_size"]}$')
ax.legend()

# Right: deviation |rho_popdyn - rho_thy| in absolute and SNR form
ax = axes[1]
sv = dev['sv_grid'][1:]
ax.semilogy(sv, np.maximum(dev['delta_abs'][1:], 1e-6), '-', color='C3', lw=1.5,
            label=r'$|\rho_\mathrm{popdyn} - \rho_\mathrm{thy}|$')
ax.semilogy(sv, np.maximum(dev['rho_popdyn_std'][1:], 1e-6), '--', color='gray', lw=1,
            label=r'popdyn MC noise (cross-$\chi$ std)')
ax.set(xlabel=r'$s$', ylabel='deviation',
       title=fr'localisation diagnostic: deviation vs s ($\alpha_\text{{Wigner-equiv}} = {0.75}$, BG intermediate)',
       ylim=(1e-5, 1.0))
ax.axhline(0.01, color='k', lw=0.5, ls=':', alpha=0.5)
ax.legend()

fig.tight_layout()
fig.savefig(FIGDIR / 'density_deviation_diagnostic_alpha1.5.png', dpi=120)
plt.show()

# Read off s_critical: first s where |dev| > both 0.01 AND 3x noise floor
delta = dev['delta_abs']
noise = dev['rho_popdyn_std']
sv_grid = dev['sv_grid']
threshold_abs = 0.01
threshold_snr = 3.0
hit_idx = None
for i in range(1, len(sv_grid)):
    if delta[i] > threshold_abs and delta[i] > threshold_snr * max(noise[i], 1e-6):
        hit_idx = i
        break
if hit_idx is not None:
    print(f'localisation onset s_c ~ {sv_grid[hit_idx]:.3f} (first s with |dev|>0.01 and SNR>3)')
else:
    print('no clear localisation onset detected within s_max')
print(f'max deviation: {np.nanmax(delta):.4f} at s = {sv_grid[np.nanargmax(delta)]:.3f}')

## 8. Pool-size scaling sweep: heavy-tail exponent $\nu(s)$

Building on Section 7: run popdyn at multiple pool sizes $P = 2^{n_d}$ for several $n_d$.  Fit $\log|\Delta_\rho(s; P)|$ vs $\log P$ at each $s$.  Slope $m$ encodes the heavy-tail index $\nu(s) = 1/(m + 1)$ of the local resolvent distribution via the generalised CLT:

| slope $m$ | $\nu = 1/(m+1)$ | regime |
|---|---|---|
| $m = -1/2$ | $\nu = 2$ | CLT-convergent (delocalised) |
| $-1/2 < m < 0$ | $1 < \nu < 2$ | heavy-tail CLT (partial localisation) |
| $m = 0$ | $\nu = 1$ | marginal -- pool-mean does not converge |
| $m > 0$ | $\nu < 1$ | divergent pool-mean (strong localisation) |

So the SV at which $m$ crosses $0$ (or equivalently $\nu$ crosses 1) is a sharp localisation onset $s_c$.  See `RMT/localisation.md` sec. 9K.

In [ ]:
sweep = htj.density_deviation_pool_sweep(
    alpha=1.5, sigma_w=1.0,
    s_max=4.0, num_points=41,
    pool_doublings_list=(4, 5, 6, 7, 8),
    num_chis=8,
    quadrature_order=64, profile_order=32,
    seed=0, num_steps_per_element=100,
)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# Left: |Delta| vs P at a few s values
ax = axes[0]
sv = sweep['sv_grid']
Ps = sweep['Ps']
for idx in [2, 8, 15, 22, 28]:
    if idx < len(sv):
        s = sv[idx]
        deltas = np.array([sweep['delta_by_P'][P][idx] for P in Ps])
        ax.loglog(Ps, np.maximum(deltas, 1e-15), 'o-', ms=5,
                  label=fr'$s={s:.2f}$, slope={sweep["slopes"][idx]:+.2f}')
Ps_ref = np.array([Ps[0], Ps[-1]], dtype=float)
ax.loglog(Ps_ref, 0.05 * (Ps_ref/Ps_ref[0])**(-0.5), 'k:', lw=1, label=r'$P^{-1/2}$ (CLT)')
ax.set(xlabel='pool size $P$', ylabel=r'$|\Delta_\rho|$',
       title=r'deviation vs pool size at fixed $s$')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Right: slope m and nu vs s
ax = axes[1]
ax2 = ax.twinx()
sv_pos = sv[1:]
slopes = sweep['slopes'][1:]
nu_vals = sweep['nu'][1:]
l1, = ax.plot(sv_pos, slopes, 'o-', color='C0', ms=4, label='slope $m$')
ax.axhline(-0.5, color='C0', ls=':', alpha=0.5)
ax.axhline(0.0, color='k', ls='-', lw=0.5, alpha=0.5)
ax.set(xlabel='$s$', ylabel='slope $m$', ylim=(-2.0, 0.5))
ax2.plot(sv_pos, np.clip(nu_vals, 0, 5), 's-', color='C3', ms=4)
ax2.axhline(2.0, color='C3', ls=':', alpha=0.5)
ax2.axhline(1.0, color='C3', ls='--', alpha=0.5)
ax2.set_ylabel(r'$\nu = 1/(m+1)$', color='C3')
ax2.set_ylim(0, 5)
ax.set_title(fr'slope and heavy-tail index $\nu(s)$, $\alpha=1.5$')

# Localisation onset s_c: where m crosses 0 (nu crosses 1)
zero_crossings = []
for i in range(len(slopes) - 1):
    if slopes[i] < 0 < slopes[i + 1] or (slopes[i] < -0.1 and slopes[i + 1] > -0.1):
        # crossing of m = -0.1 (close to 0)
        zero_crossings.append(sv_pos[i])

s_c = zero_crossings[0] if zero_crossings else float('nan')
print(f'mean slope in bulk (s in [0.2, 1.2]):  {np.nanmean(slopes[(sv_pos>0.2)&(sv_pos<1.2)]):+.3f}')
print(f'mean slope in tail (s in [2.5, 4.0]):  {np.nanmean(slopes[(sv_pos>2.5)&(sv_pos<4.0)]):+.3f}')
print(f'localisation onset s_c (first m ~ 0):  {s_c:.3f}')

fig.tight_layout()
fig.savefig(FIGDIR / 'pool_sweep_nu_alpha1.5.png', dpi=120)
plt.show()

## Scope and reductions

* $\alpha = 2$ is **not** run on the heavy-tailed code path (`belinschi_quantile_scale(2, 1) = 0`); see md sec. 4. The Gaussian limit reduces to Marchenko-Pastur with parameter $\sigma_w^2 \mathbb{E}[\phi'(h)^2]$ at the fixed point.
* For unbounded $\phi$ (ReLU) the forward $q^*$ diverges and the framework does not apply; md sec. 6 / `heavy_tailed_mlp.md` sec. 4(c).
* For $\gamma \ne 1$ (non-square layers) the atom $1 - \gamma$ at zero reappears; Theorem 2 still applies with the same scalar closure (eq. 4 of the md).
* Product Jacobian across depth is deferred to a separate derivation -- it requires free-multiplicative composition of heavy-tailed S-transforms.